In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [ ]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

In [ ]:
len(words)

In [ ]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

In [ ]:
# 测试数据集 只用前5个词
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

In [ ]:
X.shape, X.dtype, Y.shape, Y.dtype

In [ ]:
# X作为输入 Y是labels
# 现在要构建一个神经网络 将X作为输入 输出Y

# 将27个字符embed到2维空间 即27字符都有 two-dimenional embedding
C = torch.randn((27, 2))

In [ ]:
# 论文图中的index for W(t-n+1)
# 可以理解为C的下标
# 而另一种理解方式是 它就是神经网络的一个输入层 把整数下标 编码成one-hot 然后输出到后续层

In [ ]:
# 下面两种写法是一样的
C[5]
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

In [ ]:
C[[5,6,7]]
C[torch.tensor([5,6,7])]

In [ ]:
# X的大小为[32, 3] 对X中每一个元素X[m, n], 对应的embedding保存在C[X][m, n]中
emb = C[X]
emb.shape

In [ ]:
X[27, 1]

In [ ]:
C[19] == C[X][27, 1]

In [ ]:
# 下面开始构建 hidden layer
# W1的输入个数是6 (预测前3个词 每个词的embedding维度是2), neurons的数量可以随意指定, 这里用100
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [ ]:
# 本质上想做的事情如下 但因为维度对不上
# emb @ W1 + b1

In [ ]:
# 把emb的维度从[32, 3, 2]变成[32, 6]
# torch.unbind(emb, 1)得到3个[32,2]的tensor 再cat得到[32,6]
torch.cat(torch.unbind(emb, 1), 1).shape

In [ ]:
# 另一种写法更快
emb.view(32, 6)

In [ ]:
h = emb.view(32, 6) @ W1 + b1

In [ ]:
h = torch.tanh(h)

In [ ]:
h.shape

In [ ]:
# 下面开始构建 output layer
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [ ]:
# [32, 27]
logits = h @ W2 + b2
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)

In [ ]:
# the current probabilities to the correct character in the sequence,
# which is assigned by the neural network with W1/b1/W2/b2 of its weights
probs[torch.arange(32), Y]

In [ ]:
loss = -probs[torch.arange(32), Y].log().mean()
loss

In [ ]:
# 完整数据集
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

# Xtr Xdev Xte分别对应training split, dev split和test split
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])


In [ ]:
# ------------ now made respectable :) ---------------

In [ ]:
Xtr.shape, Ytr.shape # dataset

In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 200), generator=g) # 200 is neruons数量
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [ ]:
sum(p.nelement() for p in parameters) # number of parameters in total

In [ ]:
emb = C[X] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)

# 两种写法一样
# counts = logits.exp()
# probs = counts / counts.sum(1, keepdims=True)
# loss = -probs[torch.arange(32), Y].log().mean()
loss = F.cross_entropy(logits, Y)


In [ ]:
for p in parameters:
  p.requires_grad = True

In [ ]:
# [0.001, 1]区间内生成1000个数 注意指数lre是线性分布 而lrs是指数分布的
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre
# 然后通过对每个learning rate进行少次迭代 确定最优的learning rate

In [ ]:
lri = []
lossi = []
stepi = []

In [ ]:
# Perform forward pass and backward pass and update on batches of data.
# Randomly select some portion of the data set, which is called minibatch.
# 只给一部分输入，好处是梯度下降速度会更快。尽管梯度不会非常准确，即不一定再指向上升最快的方向，但大致方向仍然是对的。
# It is much better to have an approximate gradient and just make more steps than 
# it is to evaluate the exact gradient and take fewer steps.

for i in range(1000):
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (32,))
  
  # forward pass
  emb = C[Xtr[ix]] # (32, 3, 10)
  h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 200)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Ytr[ix])
  #print(loss.item())
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = lrs[i]
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  lri.append(lre[i])
  stepi.append(i)
  lossi.append(loss.log10().item())

#print(loss.item())

In [ ]:
plt.plot(lri, lossi)

In [ ]:
for i in range(20000):
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (32,))
  
  # forward pass
  emb = C[Xtr[ix]] # (32, 3, 10)
  h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 200)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Ytr[ix])
  #print(loss.item())
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 100000 else 0.01  # learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

#print(loss.item())

In [ ]:
plt.plot(stepi, lossi)

In [ ]:
emb = C[Xtr] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
emb = C[Xdev] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ydev)
loss

In [ ]:
# visualize dimensions 0 and 1 of the embedding matrix C for all characters
plt.figure(figsize=(8,8))
plt.scatter(C[:,0].data, C[:,1].data, s=200)
for i in range(C.shape[0]):
    plt.text(C[i,0].item(), C[i,1].item(), itos[i], ha="center", va="center", color='white')
plt.grid('minor')

In [ ]:
# training split, dev/validation split, test split
# 80%, 10%, 10%

In [ ]:
context = [0] * block_size
C[torch.tensor([context])].shape

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # 每次输入的样本数量是1 输入前block_size个字符 每个字符的embeding维度是d
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      # 根据概率分布采样 得到一个输出字符的下标
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))